In [ ]:
#Import everything
import matplotlib.pyplot as plt
import numpy as np
from pymatgen.core.structure import Structure, Lattice, IStructure
from pymatgen.io.vasp import Kpoints
from pymatgen.util.coord import get_angle
import matplotlib as mpl
import cmath
import TBLR2 as TBLR
import tqdm as tqdm
from matplotlib.collections import LineCollection
import matplotlib.patheffects as pe
from tqdm import tqdm
from pymatgen.io.vasp.outputs import Outcar
from sympy.matrices import Matrix
import pythtb
from pythtb import * # import TB model class
import matplotlib.pyplot as plt
import pandas as pd


In [ ]:
#Create pythTB from wannier
#dir ="/projects/SCHOOP/gc2596/Wannier_test/Diamond/wannier"
calc_dir = "/Users/gcarrel/Princeton Dropbox/Gabrielle Carrel/Laptop_Backup/SCHOOP/to_github/ORLT/CsI3"
wann_dir = calc_dir
band_dir = calc_dir+"/band"
OUTCAR = Outcar(wann_dir+"/OUTCAR")
mu = OUTCAR.efermi
# evals=my_model.sol
wannier_material=W90(wann_dir,r"wannier90")
crys_struct = Structure.from_file(wann_dir+"/POSCAR")
my_model=wannier_material.model(min_hopping_norm=0.1) # get tight-binding model without hopping terms above 0.0001 eV

my_model = my_model.make_supercell(np.array([[2,0,0],[0,2,0],[0,0,2]])) #Make_Supercell

#This works, but only with band k-points.. 
kpoints = Kpoints.from_file(band_dir+"/KPOINTS")
kpoint_list = kpoints.kpts[::2]
kpoint_list.append(kpoints.kpts[-1])
kpoint_labels=kpoints.labels[::2]
kpoint_labels.append(kpoints.labels[-1])

k_vec,k_dist,k_node=my_model.k_path(kpoint_list,nk = 301)
print(np.shape(k_vec))
evals=my_model.solve_ham(k_vec)
print(np.shape(evals))
fig, ax = plt.subplots()
for i in range(evals.shape[1]):
    ax.plot(k_dist,evals[:,i],"k-")
for n in range(len(k_node)):
    ax.axvline(x=k_node[n],linewidth=0.5, color='k')



ax.set_xlabel("")
ax.set_ylabel("E (eV)")
ax.set_xlim(k_dist[0],k_dist[-1])
ax.set_xticks(k_node)
ax.set_xticklabels(kpoint_labels)
fig.tight_layout()

In [ ]:
#Run to get comparisson DFT and wannier
OUTCAR = Outcar(wann_dir+"/OUTCAR")
mu = OUTCAR.efermi
# evals=my_model.solve_all(k_vec)
fig, ax = plt.subplots()

K_labels=pd.read_csv(band_dir+"/KLABELS",sep=r'\s+',comment="#")
band = pd.DataFrame(pd.read_csv(band_dir+"/REFORMATTED_BAND.dat",sep=r'\s+',comment='#'))


for i in band.columns[1:]:
    ax.plot(band[band.columns[0]]*(k_dist[-1]/np.array(band[band.columns[0]])[-1]),band[i],color="#3953a4")
#ax.set_xlim(list(K_labels['Coordinate'])[0],list(K_labels['Coordinate'])[-1])

for i in range(evals[0].shape[0]):
    ax.plot(k_dist,evals[:,i]-mu,color="#faa316",linestyle="dashed")
for n in range(len(k_node)):
    ax.axvline(x=k_node[n],linewidth=0.5, color='k')


ax.set_xlabel("")
ax.set_ylabel(r"$E-E_f \quad(eV)$")
ax.set_xlim(k_dist[0],k_dist[-1])
ax.set_xticks(ticks=K_labels['Coordinate']*(k_dist[-1]/np.array(band[band.columns[0]])[-1]),labels=kpoint_labels)

plt.gcf().set_size_inches(6,4)
plt.ylabel(r"E-E$_f$ (eV)")
fig.tight_layout()
#plt.savefig(wann_dir+"/Wannier_DFT_compare.png",dpi=400,bbox_inches='tight')  




In [ ]:
#Create array with all hopping information
H = np.empty((my_model.norb,my_model.norb), dtype=object)
for i in range(len(H)):
    for j in range(len(H)):
        H[i,j] = []
orb_pos = my_model.orb_vecs
abs_dist_list = []
t_list=[]
for hop in my_model.hoppings:
    #print("hop",hop)
    if len(hop)==3:
        t = hop['amplitude']
        i = hop['from_orbital']
        j = hop['to_orbital']
        j_image = np.array([0,0,0])
        #print(t,i,j,j_image)
    else:
        t = hop['amplitude']
        i = hop['from_orbital']
        j = hop['to_orbital']
        j_image = hop['lattice_vector']
       # print(t,i,j,j_image)
    dist = ((orb_pos[j]+j_image)-orb_pos[i])@my_model.lattice.lat_vecs
    abs_dist = np.linalg.norm(dist)
    abs_dist_list.append(abs_dist)
    t_list.append(t)
    H[i,j].append([t,dist[0],dist[1],dist[2]])

In [ ]:
#Function that finds hamiltonian from hopping array at specific k-point
def ham_func(k):
    kx,ky,kz=k
    H_res = np.zeros((my_model.norb,my_model.norb),dtype = complex)
    for i in np.arange(0,my_model.norb,1):
        for j in np.arange(0,my_model.norb):
            if len(H[i][j]) == 0:
                pass 
            else:
                for item in H[i][j]:
                    item0=complex(item[0])
                    item1=complex(item[1])
                    item2=complex(item[2])
                    item3=complex(item[3])
                    H_res[i][j] += np.round(item0,10)*(np.exp(1j* kx *item1+ 1j* ky *item2 +1j* kz *item3))
                    H_res[j][i] += np.conj(np.round(item0,10)*(np.exp(1j* kx *item1+ 1j* ky *item2 +1j* kz *item3)))
    for i,onsite in enumerate(my_model._site_energies):
        H_res[i,i]+=np.round(onsite,10)
    return H_res

In [ ]:
#Make tight-binding object in Nish's code
#CHANGE THIS BACK TO 15x15x15
nkpx=15#set to 15 for 2x2x2
nkpy=15
nkpz=15
lat = my_model.get_lat()
tb = TBLR.TBLinearResponse(ham_func, Lar=np.array(lat), 
                           Nkp=[nkpx,nkpy,nkpz],    # the first entry is number of points for line cuts and second entry is mesh size (200x200)
                           prestr='NAME',   # this is an preffix identifier for filenames -- be sure to change it when changing parameters!
                           save_all=False,   # if true, it will i/o data from dataEnUm/ -- it speeds up the code!
                           kpath = ['G', 'X', 'M', 'G','Y','M'],   # high symmetry path for band structure plots   
                           )
#occ=4*2*2*2#int(1) #Change this based on lattice (automate using pymatgen??)
occ = int(len(tb.enUM[0,:,0][tb.enUM[0,:,0]<=mu]))

print("mu",mu)
print("occ",occ)

#This runs with band structure directory
path=kpoint_list
klabels=kpoint_labels
evals, evecs,klabels= tb.solve_alongpath3D(path=path) 
evecs = np.array(evecs)
plt.plot(evals, 'k',linewidth=2)

k_index = np.arange(0,np.shape(evals)[0])

plt.xlim(klabels[0],klabels[-1])
plt.ylabel(r'$E (eV)$')
for k in klabels:
    plt.axvline(k,color="grey")
plt.axhline(mu,linestyle="dashed",color="#852D1F",linewidth=2)
plt.tight_layout()
plt.gcf().set_size_inches(4, 4)
#plt.savefig(wann_dir+"/2x2x2_band.png",dpi=400,bbox_inches='tight')  


In [ ]:
qgtxx = tb.solve_QGT_kresolved(mu=mu,dkx=np.array([1,0,0]))
print(np.sum(qgtxx)/(len(tb.kSpan)*occ))

In [ ]:
#INCLUDING MORE FOR LOOPS, NOT k-resolved(took too much memory)
import opt_einsum as oe

mu=mu 
beta_const = 1.e2			# smearing for Fermi factors
dkx = np.array([1,0,0])	    # direction for taking k derivative
dky = np.array([0,1,0])
k_step = 1.e-3
FS=False
delta=1.e-3							# infinitesimal step
nkp=len(tb.kSpan)
#Jop=np.zeros((nkp,tb.norb,tb.norb)) #size K,m,n
Jop1=[]
Jop2 = []
def Fermi(beta_const, x):
	return 1.0/(1.0 + np.exp(beta_const*x))
denom = (len(tb.kSpan)*occ)
def FNM(em, en):
    delta = 1.e-3  
    emn = em - en
    fn = Fermi(beta_const, en)
    fm = Fermi(beta_const, em)
    fnm = fn*(1.0 - fm)
    if FS:
        result = np.where(np.abs(emn) < delta, -DFermi(beta_const, em), fnm / (emn**2))
    else:
        result = np.where(np.abs(emn) < delta, 0.0, fnm / (emn**2))
    return result

# Precompute 'em' and 'en' since they do not depend on 'time'
def calc_QM():
    em = tb.enUM[:, :, 0] - mu
    en = tb.enUM[:, :, 0] - mu
    em = em[:, :, np.newaxis]  # Shape (num_i, norb, 1)
    en = en[:, np.newaxis, :]  # Shape (num_i, 1, norb)
    
    pref = FNM(en, em)  # vectorized FNM
    qgtxx=0
    qgtxy=0
    qgtyx=0
    qgtyy=0
    		# Reshape for broadcasting to compute differences
    em = em[:, :, np.newaxis]  # Shape (num_i, norb, 1)
    en = en[:, np.newaxis, :]  # Shape (num_i, 1, norb)
    
    qgtxx=0
    
    for i,k in enumerate( tqdm(tb.kSpan )):
        Hk = tb.ham(k)
        um = tb.enUM[i,:,1:]
        Hk_dkx = tb.ham(k + k_step*dkx)
        Hk_dky = tb.ham(k + k_step*dky)
        dHx = (Hk_dkx - Hk)/k_step
        dHy = (Hk_dky - Hk)/k_step
        um = np.asmatrix(um)
        uH = um.H
        Jv1x = oe.contract("ia,ad,dj->adij",uH,dHx,um)
        Jv2x = oe.contract("ic,cb,bj->bcij",uH,dHx,um)
        Jv1y = oe.contract("ia,ad,dj->adij",uH,dHy,um)
        Jv2y = oe.contract("ic,cb,bj->bcij",uH,dHy,um)
        Jv2x_conj = np.conj(Jv2x)
        Jv2y_conj = np.conj(Jv2y)
        qgtxx += oe.contract("ij,adij,bcij->abcd",pref[i],Jv1x,Jv2x_conj)
        qgtxy += oe.contract("ij,adij,bcij->abcd",pref[i],Jv1x,Jv2y_conj)
        qgtyx += oe.contract("ij,adij,bcij->abcd",pref[i],Jv1y,Jv2x_conj)
        qgtyy += oe.contract("ij,adij,bcij->abcd",pref[i],Jv1y,Jv2y_conj)
    return qgtxx/denom,qgtxy/denom,qgtyx/denom,qgtyy/denom


In [ ]:

qgtxx,qgtxy,qgtyx,qgtyy = calc_QM()
np.save(wann_dir+'QM_orb_res_qgtxx.npy', qgtxx)
np.save(wann_dir+'QM_orb_res_qgtyy.npy', qgtyy)
np.save(wann_dir+'QM_orb_res_qgtxy.npy', qgtxy)
np.save(wann_dir+'QM_orb_res_qgtyx.npy', qgtxx)

In [ ]:
#LOAD FROM HERE ... QM_orb_res_qgtxx.npy

qgtxx=np.load('/Users/gcarrel/Princeton Dropbox/Gabrielle Carrel/Laptop_Backup/SCHOOP/to_github/ORLT/CsI3/QM_orb_res_qgtxx.npy')
qgtyy=np.load('/Users/gcarrel/Princeton Dropbox/Gabrielle Carrel/Laptop_Backup/SCHOOP/to_github/ORLT/CsI3/QM_orb_res_qgtyy.npy')
qgtxy=np.load('/Users/gcarrel/Princeton Dropbox/Gabrielle Carrel/Laptop_Backup/SCHOOP/to_github/ORLT/CsI3/QM_orb_res_qgtxy.npy')
qgtyx=np.load('/Users/gcarrel/Princeton Dropbox/Gabrielle Carrel/Laptop_Backup/SCHOOP/to_github/ORLT/CsI3/QM_orb_res_qgtyx.npy')

In [ ]:

import itertools

def canonicalize(combo):
    mapping = {}
    next_label = ord('a')
    result = []
    for x in combo:
        if x not in mapping:
            mapping[x] = chr(next_label)
            next_label += 1
        result.append(mapping[x])
    return ''.join(result)
import itertools

def find_combo_list(orb_list, num_un):
    patterns = set()
    for combo in itertools.product(orb_list, repeat=4):
        if len(set(combo)) >= num_un:
            patterns.add(canonicalize(combo))

    return sorted(patterns)
    
def find_atom_metric(atomic_metric,num_un):
    atomic_metric_ab = atomic_metric.copy()
    for a in range(len(atoms)):
        for b in range(len(atoms)): 
            for c in range(len(atoms)):
                indices = [a,b,c][:num_un]
                #print("indices",indices)
                direct_indices = np.asarray(list(itertools.product(indices,repeat=4)))
                for ind in direct_indices:
                    atomic_metric_ab[ind[0],ind[1],ind[2],ind[3]] = 0
    return atomic_metric_ab

combo_a = find_combo_list(["a"],1)
combo_ab = find_combo_list(["a","b"],2)
combo_abc = find_combo_list(["a","b","c"],3)
combo_abcd = find_combo_list(["a","b","c","d"],4)


atoms = []
##For CsI3
#for n in np.arange(int(tb.norb/6)): #Need to change based on material
#    atom1=[int(n*6),int(n*6+1),int(n*6+2),int(n*6+3),int(n*6+4),int(n*6+5)]
#    atoms.append(atom1)
#For BaTiO3
#atoms = []
#for n in np.arange(int(tb.norb)/20):
#    atom1=[int(n*20),int(n*20+1),int(n*20+2),int(n*20+3),int(n*20+4),int(n*20+5)]
#    atom2=[int(n*20+6),int(n*20+7),int(n*20+8),int(n*20+9),int(n*20+10)]
#    atom3=[int(n*20+11),int(n*20+12),int(n*20+13),int(n*20+14),int(n*20+15),int(n*20+16),int(n*20+17),int(n*20+18),int(n*20+19)]
#    atoms.append(atom1)
#    atoms.append(atom2)
#    atoms.append(atom3)
#For GeTe,AlP
#atoms = []
#for n in np.arange(int(tb.norb)/4):
#    atom=[int(n*4),int(n*4+1),int(n*4+2),int(n*4+3)]
#    atoms.append(atom)
#for NaCl
#atoms = []
#for n in np.arange(int(tb.norb)/7):
#    atom1=[int(n*7),int(n*7+1),int(n*7+2),int(n*7+3)]
#    atom2=[int(n*7+4),int(n*7+5),int(n*7+6)]
#    atoms.append(atom1)
#    atoms.append(atom2)
#print("atoms",atoms)

##for PbS
#atoms = []
#for n in np.arange(int(tb.norb)/3):
#    atom1=[int(n*3),int(n*3+1),int(n*3+2)]
#    atoms.append(atom1)
print("atoms",atoms)

atomic_metric = np.zeros((len(atoms),len(atoms),len(atoms),len(atoms)))
for i,m in enumerate(atoms): 
    for j,n in enumerate(atoms):
        for k,o in enumerate(atoms):
            for l,p in enumerate(atoms):
                atomic_metric[i,j,k,l] = np.sum(qgtxx[np.ix_(m,n,o,p)]+qgtyy[np.ix_(m,n,o,p)]) 


atomic_metric = np.zeros((len(atoms),len(atoms),len(atoms),len(atoms)))
for i,m in enumerate(atoms): 
    for j,n in enumerate(atoms):
        for k,o in enumerate(atoms):
            for l,p in enumerate(atoms):
                atomic_metric[i,j,k,l] = np.sum(qgtxx[np.ix_(m,n,o,p)]+qgtyy[np.ix_(m,n,o,p)])  
                

atom_metric_a = find_atom_metric(atomic_metric,0)
atom_metric_ab = find_atom_metric(atomic_metric,1)
atom_metric_abc = find_atom_metric(atomic_metric,2)
atom_metric_abcd = find_atom_metric(atomic_metric,3)

data1 = []
data2 = []
data3 = []
data4 = []
#Find quantum metric one orbitals
denom=1
for combo in combo_a:
    data1.append([str(combo),np.float64(np.sum(np.einsum(combo+"->",atomic_metric))/denom)])

#Find quantum metric two orbitals                
for combo in np.setdiff1d(combo_ab,combo_a):
    data2.append([str(combo),np.float64(np.sum(np.einsum(combo+"->",atom_metric_ab))/denom)])
#Find quantum metric three orbitals
for combo in np.setdiff1d(combo_abc,combo_ab):
    data3.append([str(combo),np.float64(np.sum(np.einsum(combo+"->",atom_metric_abc))/denom)])
#Find quantum metric four orbitals:
for combo in np.setdiff1d(combo_abcd,combo_abc):
    data4.append([str(combo),np.float64(np.sum(np.einsum(combo+"->",atom_metric_abcd))/denom)])

#Find Quantum Metric four  orbitals
data1=np.array(data1)
data2=np.array(data2)
data3=np.array(data3)
data4=np.array(data4)
print(np.sum(qgtxx+qgtyy)/denom)



In [ ]:
import pandas as pd
concat = np.concatenate((data1[:,1],data2[:,1],data3[:,1],data4[:,1]))
print(np.min(np.float64(concat)))
df1 = pd.DataFrame(data=data1,columns = ['1orb','QM1orb'])
df2 = pd.DataFrame(data=data2,columns = ['2orb','QM2orb'])
df3 = pd.DataFrame(data=data3,columns = ['3orb','QM3orb'])
df4 = pd.DataFrame(data=data4,columns = ['4orb','QM4orb'])

df = pd.concat([df1, df2,df3,df4], axis=1) 
df['QM1orb'] = np.float64(df['QM1orb'])
df['QM2orb'] = np.float64(df['QM2orb'])
df['QM3orb'] = np.float64(df['QM3orb'])
df['QM4orb'] = np.float64(df['QM4orb'])


styler = df.style.background_gradient(cmap='viridis',subset=['QM1orb','QM2orb','QM3orb','QM4orb'],vmin=np.min(np.float64(concat)),vmax=np.max(np.float64(concat))).highlight_null(color="transparent")
with pd.ExcelWriter(wann_dir+'/NAME.xlsx') as writer:  
    styler.to_excel(writer,engine='xlsxwriter',index=False)
print(wann_dir)

